# NCDD OCR — Netra-OCR

Process one PDF with Netra-OCR, save TXT, and extract articles as JSON and CSV.

In [36]:
%pip install "netra-ocr[full]"


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
import csv
import os
import json
import re
from pathlib import Path
import torch
from netra_ocr.exporters import save_document
from netra_ocr.ocr_engine import KhmerOCRPipeline
from tqdm.auto import tqdm

In [38]:
INPUT_DIR = Path('ncdd_laws/pdfs')
INPUT_PDF = INPUT_DIR / "17040.pdf"
OUTPUT_DIR = Path('ncdd_ocr_netra')
TEXT_DIR = OUTPUT_DIR / 'text'
ARTICLE_JSON_DIR = OUTPUT_DIR / 'articles' / 'json'
ARTICLE_CSV_DIR = OUTPUT_DIR / 'articles' / 'csv'
NETRA_CONF = 0.25
NETRA_PAD = 4
NETRA_BEAM_WIDTH = 1
NETRA_BATCH_SIZE = 8
USE_MPS = False  # Set True only after confirming MPS works for your PyTorch build.
NETRA_DEVICE = 'mps' if USE_MPS and torch.backends.mps.is_available() else 'cpu'
os.environ['NETRA_DEVICE'] = NETRA_DEVICE
for folder in (TEXT_DIR, ARTICLE_JSON_DIR, ARTICLE_CSV_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if not INPUT_PDF.exists():
    available = sorted(INPUT_DIR.glob('*.pdf'))
    if not available: raise FileNotFoundError(f'No PDF files found in {INPUT_DIR}')
    INPUT_PDF = available[0]
print('Test document:', INPUT_PDF)
print('Netra device:', NETRA_DEVICE)

Test document: ncdd_laws/pdfs/17040.pdf
Netra device: cpu


In [39]:
ARTICLE_PATTERNS = (re.compile(r'^\s*(?:មាត្រា|មាត្រាទី)\s*([០-៩0-9]+)\s*[:.\-]?\s*(.*)$', re.IGNORECASE), re.compile(r'^\s*article\s*([0-9]+)\s*[:.\-]?\s*(.*)$', re.IGNORECASE))
def clean_text(text): return re.sub(r'\s+', ' ', text).strip()
def extract_articles(text):
    articles, current = [], None
    for line in text.splitlines():
        line = clean_text(line)
        if not line: continue
        heading = next((p.match(line) for p in ARTICLE_PATTERNS if p.match(line)), None)
        if heading:
            if current: current['text'] = clean_text(' '.join(current.pop('_parts'))); articles.append(current)
            current = {'article_number': heading.group(1), 'title': clean_text(heading.group(2)), 'text': '', '_parts': []}
        elif current: current['_parts'].append(line)
    if current: current['text'] = clean_text(' '.join(current.pop('_parts'))); articles.append(current)
    return articles
def save_articles(articles, document_id):
    json_path = ARTICLE_JSON_DIR / f'{document_id}_articles.json'; csv_path = ARTICLE_CSV_DIR / f'{document_id}_articles.csv'
    with json_path.open('w', encoding='utf-8') as file: json.dump(articles, file, ensure_ascii=False, indent=2)
    fields = ['article_number', 'title', 'text']
    with csv_path.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields); writer.writeheader(); writer.writerows({f: a.get(f, '') for f in fields} for a in articles)
    return json_path, csv_path

In [40]:
def process_one_document(pdf_path=INPUT_PDF, overwrite=False):
    pdf_path = Path(pdf_path); document_id = pdf_path.stem
    text_path = TEXT_DIR / f'{document_id}.txt'; json_path = ARTICLE_JSON_DIR / f'{document_id}_articles.json'; csv_path = ARTICLE_CSV_DIR / f'{document_id}_articles.csv'
    if not overwrite and all(path.exists() for path in (text_path, json_path, csv_path)): return {'document': pdf_path.name, 'status': 'skipped'}
    text_path.parent.mkdir(parents=True, exist_ok=True)
    pipeline = KhmerOCRPipeline(
        detector='yolo', conf=NETRA_CONF, pad=NETRA_PAD,
        layout=True, decoder='blockwise',
    )
    document = pipeline.process_document(
        str(pdf_path), beam_width=NETRA_BEAM_WIDTH,
        batch_size=NETRA_BATCH_SIZE, include_headers_footers=True,
    )
    save_document(document, str(text_path))
    text = text_path.read_text(encoding='utf-8'); articles = extract_articles(text); save_articles(articles, document_id)
    return {'document': pdf_path.name, 'status': 'success', 'articles': len(articles), 'text': str(text_path)}
result = process_one_document()
print(result)

Initializing detector: yolo
Loading YOLO weights: /Users/nak/CADT/CADT_Fourth_Year/info-retrival/.venv/lib/python3.14/site-packages/netra_ocr/detectors/yolo26s_best/best.pt


Batch prediction error; retrying one image at a time
Traceback (most recent call last):
  File "/Users/nak/CADT/CADT_Fourth_Year/info-retrival/.venv/lib/python3.14/site-packages/netra_ocr/recognition/recognize_text.py", line 111, in recognize_batch
    return predictor.predict_batch(image_list, beam_width=beam_width, batch_size=batch_size)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nak/CADT/CADT_Fourth_Year/info-retrival/.venv/lib/python3.14/site-packages/netra_ocr/recognition/predictor.py", line 227, in predict_batch
    f = self.model.cnn(batch_tensor)
  File "/Users/nak/CADT/CADT_Fourth_Year/info-retrival/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/nak/CADT/CADT_Fourth_Year/info-retrival/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py", line 1794, in _

KeyboardInterrupt: 

In [ ]:
def process_all_documents(input_dir=INPUT_DIR, overwrite=False):
    results = []
    for pdf_path in tqdm(sorted(Path(input_dir).glob('*.pdf')), desc='Netra-OCR documents'):
        try: results.append(process_one_document(pdf_path, overwrite))
        except Exception as error: results.append({'document': pdf_path.name, 'status': 'failed', 'error': str(error)})
    return results
# all_results = process_all_documents()